First, get to the correct working directory and activate the virtual environment for the demo:

In [ ]:
cd $BASEDIR
source conda/etc/profile.d/conda.sh
conda activate demo

Look at the `uwtools`-enabled configuration file for the demo. Don't panic: There's a lot to see, but we'll examine various blocks in more detail as we go. One thing to note looking at the full config is the use of [Jinja2](https://jinja.palletsprojects.com/en/stable/templates/) expressions to correctly tie together various config blocks.

In [ ]:
cat config.yaml

The `ecflow.server` block provides the bare mininum for running the ecFlow server: A definition of the `ECF_HOME` environment variable, which specifies where the server should run and where, by default, it will look for `.h` include files when creating job scripts from `.ecf` scripts:

In [ ]:
uw config realize -i config.yaml --key-path ecflow.server

In this case, `ecflow.server` refers to `app.basedir` for its value, and `app.basedir` refers to the environment variable `BASEDIR` set by the `start` script.

Start the ecFlow server using the config specified in `config.yaml`. Request a JSON-formatted report of important server environment variables, redirect that to `server.json`, and redirect the rest of the output to `server.log`:

In [ ]:
uw ecflow server -c config.yaml --report >server.json 2>server.log &

Wait until the server has started up and written `server.json`:

In [ ]:
while [[ ! -s server.json ]]; do sleep 1; done

View `server.log` and note that there were no errors, and the the server is using a random available TCP port.

In [ ]:
cat server.log

At server-startup time, `uw ecflow server` also creates a `.h` include file called `server.h`, defining environment variables that will be needed by `.ecf` scripts:

In [ ]:
cat server.h

The `server.json` file, containing current values for some useful server-related environment variables, was also created:

In [ ]:
jq . server.json

Use `jq` to extract values from this file to use to set some crucial environment variables that will allow the ecFlow client tool, `ecflow_client`, to successfully communicate with the server:

In [ ]:
export ECF_PORT=$(jq -r .ECF_PORT server.json)
export ECF_SSL=$(jq -r .ECF_SSL server.json)

In [ ]:
ecflow_client --ping

In [ ]:
ecflow_client --stats | grep Status

In [ ]:
ecflow_client --restart

In [ ]:
ecflow_client --stats | grep Status

In [ ]:
uw ecflow realize -c config.yaml --output-dir=. --scripts-dir=.

In [ ]:
cat suite.def

In [ ]:
tree vx

In [ ]:
cat vx/fetch_truth.ecf

In [ ]:
cat vx/extract_grids.ecf

In [ ]:
ecflow_client --load=suite.def

In [ ]:
ecflow_client --get

In [ ]:
ecflow_client --begin=vx